# **Import des modules nécessaires**

In [ ]:
import pandas as pd #pour la manipulation de données
from pathlib import Path #pour la gestion des chemins de fichiers

from bertopic import BERTopic #pour la modélisation de sujets

from umap import UMAP #pour la réduction de dimensionnalité

from sklearn.cluster import KMeans #pour le clustering
from hdbscan import HDBSCAN #pour le clustering de BERTopic

from sklearn.feature_extraction.text import CountVectorizer #pour la vectorisation de texte
from bertopic.vectorizers import ClassTfidfTransformer #pour la vectorisation de texte spécifique à BERTopic

from sentence_transformers import SentenceTransformer #pour les embeddings de phrases

# **Chargement du corpus de Zola**


In [69]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "02_corpus_zola.csv", encoding="utf-8",)
df.head()


,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",985
1,1865 La confession de Claude.,1865,1,2,"Ai-je eu trop de confiance en ma force, ma pla...",1067
2,1865 La confession de Claude.,1865,1,3,Hélas! il me faut cependant une ombre de réali...,894
3,1865 La confession de Claude.,1865,1,4,Elle dormait. J’ai entassé sur ses pieds tous ...,1158
4,1865 La confession de Claude.,1865,1,5,"Ainsi, jamais mon cœur ne pourra battre sans q...",1027


In [70]:
df.shape

(3556, 6)

# **Traitement du Corpus de Zola**

## **1) Choix du modèle d'embedding**

Ici je vais choisir un modèle d'embedding pré-entraîné pour transformer les textes en vecteurs numériques. Je vais utiliser un modèle de la bibliothèque Sentence Transformers, qui est compatible avec BERTopic.

In [71]:
embedding_model = SentenceTransformer(
    "sentence-transformers/distiluse-base-multilingual-cased-v2"
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

## **2) Pipeline de Traitement**

In [72]:
docs = df["texte"].tolist()

timestamps = df["ordre_romans"].tolist()

### 1) UMAP

In [85]:
umap_model = UMAP(
    n_neighbors=10,
    n_components=10,
    min_dist=1,
    metric="cosine",
    random_state=42
)

### 2) HDBSCAN

In [ ]:
hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="leaf",
    prediction_data=True
)

In [74]:
cluster_model = KMeans(
    n_clusters=15,
    random_state=42,
    n_init="auto"

)

### 3) creation des stop words + vectorisation TF-IDF

In [75]:
stop_word = [
    # Rougon / Macquart / noms familiaux centraux
    "rougon", "macquart", "saccard", "mouret", "lantier", "coupeau", "maheu", "fouan", "quenu", "chanteau", "josserand", "deberle",
    "raquin", "lorilleux", "buteau", "roubaud", "baudu", "gervaise",

    # Prénoms / personnages très fréquents
    "pierre", "jean", "jacques", "marie", "madeleine", "pauline", "hélène", "jeanne", "claude", "marius", "lazare", "guillaume",
    "laurent", "philippe", "thérèse", "albine", "maurice", "marthe", "maxime", "daniel", "camille", "serge", "octave", "florent",
    "félicité", "louise", "silvère", "caroline", "lisa", "laurence", "miette", "simon", "josine", "clotilde", "véronique", "juliette",
    "rosalie", "mathieu", "charles", "françoise", "lise", "séverine", "henriette", "georges", "angélique", "adèle", "pascal", "lucie", 
    "martine", "virginie", "suzanne", "valentine", "olympe", "lucien", "armande", "constance", "berthe", "sidonie", "benedetta",

    # Noms propres / personnages complémentaires
    "luc", "geneviève", "catherine", "chaval", "adélaïde", "antoine", "auguste", "marianne", "clorinde", "normande", "saget", "mlle saget",
    "delhomme", "ragu", "trouche", "sandoz", "mahoudeau", "bonneville", "rastoil", "rambaud", "fenil", "surin", "malignon", "faloise",
    "lambesc", "mignot", "baugé", "morange", "bouchard", "teuse", "pâquerette", "paquerette", "duparque", "mme duparque", "guersaint",
    "aurélie", "aurelie", "mme aurélie", "mme aurelie", "lecœur", "lecoeur", "mme lecœur", "mme lecoeur", "granoux", "bourron",
    "lerat", "mme lerat", "duveyrier", "bachelard", "gueulin", "delaherche", "beauclair", "lison", "orlando", "delbos",
    "boisgelin", "fernande", "sébastien", "sebastien", "mazaud", "dubuche", "moreux", "vigneron", "bourrette", "satan",
    "archangias", "férat", "ferat", "barbue", "trouille", "laveuve",

    # Personnages secondaires / noms très discriminants
    "fauchery", "muffat", "bordenave", "zoé", "mignon", "faujas", "mathéus", "trublot", "fagerolles", "cazalis", "poizat", "jordan",
    "vandeuvres", "condamin", "paloque", "charbonnel", "gilquin", "marjolin", "campardon", "toussaint", "grivet", "denizet", "kahn",
    "hutin", "bonnaire", "laboque", "marsy", "nanet", "favier", "deloche", "goujet", "cadine", "chouteau", "rochas", "lepailleur",
    "séguin", "gagnière", "hourdequin", "macqueron", "théodose", "savin", "gérard", "larsonneau", "morfain", "lorin", "salvat",
    "boche", "gavard", "martelly",

    # Groupes nominaux religieux / institutionnels
    "sœur hyacinthe", "soeur hyacinthe", "mère chantemesse", "mere chantemesse",

     # Formes d’adresse / famille
    "monsieur", "madame", "mademoiselle", "mme", "mlle",
    "maman", "papa", "père", "pere", "mère", "mere",
    "frère", "frere", "frères", "freres", "sœur", "soeur",
    "sœurette", "soeurette", "tante", "oncle", "cousin",
    "cousine", "époux", "epoux", "épouse", "epouse",
    "nièce", "niece",

    # Mots très fréquents et peu informatifs
    "le", "la", "les", "un", "une", "des", "du", "de", "d", "au", "aux",
    "ce", "cet", "cette", "ces", "son", "sa", "ses", "leur", "leurs",
    "je", "tu", "il", "elle", "nous", "vous", "ils", "elles",
    "me", "te", "se", "moi", "toi", "lui", "eux",
    "que", "qui", "quoi", "dont", "où", "et", "ou", "mais", "donc",
    "or", "ni", "car", "ne", "pas", "plus", "moins", "très",
    "dans", "sur", "sous", "avec", "sans", "pour", "par", "comme",
    "chez", "vers", "entre", "contre", "depuis", "pendant",
    "tout", "tous", "toute", "toutes", "même", "autre", "autres",
    "être", "avoir", "faire", "dire", "aller", "voir", "savoir",
    "pouvoir", "vouloir", "falloir", "devoir", "venir", "prendre",
    "mettre", "donner", "trouver", "passer", "rester", "sembler",
    "encore", "toujours", "jamais", "bien", "mal", "peu", "assez",
    "alors", "puis", "enfin", "aussi", "ainsi", "déjà", "là",
    "ici", "cela", "ça", "ceci", "celui", "celle", "ceux", "celles"
]

In [90]:
vectorizer_model = CountVectorizer(
    stop_words=stop_word,
    min_df=3,
    max_df=0.90
)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True
)

### 4) Création du modèle BERTopic

In [91]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    #hdbscan_model=cluster_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

df["topic"] = topics

2026-06-02 16:55:37,715 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/112 [00:00<?, ?it/s]

2026-06-02 16:56:00,635 - BERTopic - Embedding - Completed ✓
2026-06-02 16:56:00,637 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-02 16:56:10,657 - BERTopic - Dimensionality - Completed ✓
2026-06-02 16:56:10,659 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-02 16:56:10,843 - BERTopic - Cluster - Completed ✓
2026-06-02 16:56:10,847 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-02 16:56:12,065 - BERTopic - Representation - Completed ✓


## **3) Topics Présent**

In [93]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1968,-1_abbé_saint_bruit_docteur,"[abbé, saint, bruit, docteur, marc, bête, prêt...","[La montée était longue, des quartiers surgiss..."
1,0,1511,0_abbé_saint_église_bruit,"[abbé, saint, église, bruit, prêtre, rouge, fo...","[En face de l’entrée, à la place ordinaire de ..."
2,1,40,1_nana_fontan_lucy_comte,"[nana, fontan, lucy, comte, labordette, satin,...",[Les hommes qui se plantaient devant les affic...
3,2,22,2_fine_joseph_barricades_armateur,"[fine, joseph, barricades, armateur, marseille...",[Le comte s’établit sur le palier du troisième...
4,3,15,3_chantebled_moulin_docteur_nicolas,"[chantebled, moulin, docteur, nicolas, santerr...","[Ensuite, par rang de générations, les autres ..."


In [ ]:
for topic_id in topic_info["Topic"].head(15):
    if topic_id != -1:
        print("\nTOPIC", topic_id)
        print(topic_model.get_topic(topic_id)[:15])


TOPIC 0
[('millions', np.float64(0.004175623079363673)), ('bourse', np.float64(0.004089654832029945)), ('cent mille', np.float64(0.0033575832726324547)), ('cents francs', np.float64(0.0033106341423296617)), ('capital', np.float64(0.0031461236697110967)), ('cinquante', np.float64(0.0028821650804342607)), ('actions', np.float64(0.00281238609143753)), ('payer', np.float64(0.0027351951819240663)), ('ouvriers', np.float64(0.0026400789745479342)), ('hérédité', np.float64(0.002627439136523141))]

TOPIC 1
[('général', np.float64(0.00864733267784234)), ('troupes', np.float64(0.007183862267088316)), ('armée', np.float64(0.006955321928697617)), ('soldats', np.float64(0.006361377991876613)), ('sedan', np.float64(0.006191676842745636)), ('canon', np.float64(0.005818917370790989)), ('insurgés', np.float64(0.005368156016708426)), ('drapeau', np.float64(0.004782319807285889)), ('bazeilles', np.float64(0.004744506236714055)), ('ducrot', np.float64(0.004634195697380423))]

TOPIC 2
[('amant', np.float64